In [ ]:
"""
Save as: src/visualization/equity_curve_animated_fixed.py
Run: python src/visualization/equity_curve_animated_fixed.py
"""

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Create directories
VIZ_PATH = Path("visualizations/equity")
VIZ_PATH.mkdir(parents=True, exist_ok=True)
HTML_PATH = VIZ_PATH / 'html'
HTML_PATH.mkdir(parents=True, exist_ok=True)

# Professional color palette (all keys defined)
COLORS = {
    'strategy': '#2ECC71',
    'benchmark': '#3498DB',
    'drawdown': '#E74C3C',
    'positive': '#2ECC71',
    'negative': '#E74C3C',
    'neutral': '#95A5A6',
    'sharpe_positive': '#27AE60',
    'sharpe_negative': '#C0392B',
    'primary': '#1E88E5',
    'secondary': '#DC143C',
    'success': '#2ECC71',
    'warning': '#F39C12',
    'info': '#00ACC1',  # Added missing 'info' key
    'purple': '#9B59B6',
    'orange': '#E67E22',
    'dark': '#2C3E50',
    'light': '#ECF0F1'
}


def generate_animated_equity_curve(y_true, y_pred, title="S&P 500 Model Performance Dashboard", 
                                    save_path=None, benchmark=None):
    """
    Generate interactive animated equity curve dashboard
    
    Parameters:
    - y_true: actual returns array
    - y_pred: predicted returns array
    - title: chart title
    - save_path: path to save the plot
    - benchmark: benchmark returns (optional, defaults to y_true)
    """
    print("\n📈 Generating animated equity curve dashboard...")
    
    # Convert to pandas Series for easier manipulation
    y_true = pd.Series(y_true)
    y_pred = pd.Series(y_pred)
    
    # Use benchmark or actual as benchmark
    if benchmark is None:
        benchmark = y_true
    else:
        benchmark = pd.Series(benchmark)
    
    # Calculate cumulative returns
    cumulative_strategy = (1 + y_pred).cumprod()
    cumulative_benchmark = (1 + benchmark).cumprod()
    
    # Calculate drawdown using pandas
    running_max = cumulative_strategy.expanding().max()
    drawdown = (cumulative_strategy - running_max) / running_max * 100
    
    # Calculate rolling Sharpe ratio
    window = 20
    rolling_returns = y_pred.rolling(window)
    rolling_sharpe = rolling_returns.mean() / rolling_returns.std() * np.sqrt(252)
    
    # Calculate performance metrics
    total_return_strategy = (cumulative_strategy.iloc[-1] - 1) * 100
    total_return_benchmark = (cumulative_benchmark.iloc[-1] - 1) * 100
    sharpe_ratio = y_pred.mean() / y_pred.std() * np.sqrt(252)
    max_drawdown = drawdown.min()
    win_rate = (np.sign(y_pred) == np.sign(benchmark)).mean() * 100
    
    # Create time indices
    dates = pd.date_range(start='2016-01-01', periods=len(y_true), freq='W')
    
    # Create subplots
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=('Cumulative Returns Comparison', 'Rolling Sharpe Ratio (20-day)',
                       'Drawdown Analysis', 'Return Distribution',
                       'Prediction vs Actual Scatter', 'Error Distribution'),
        vertical_spacing=0.1,
        horizontal_spacing=0.12,
        row_heights=[0.35, 0.35, 0.3],
        specs=[[{'type': 'scatter'}, {'type': 'scatter'}],
               [{'type': 'scatter'}, {'type': 'histogram'}],
               [{'type': 'scatter'}, {'type': 'histogram'}]]
    )
    
    # 1. Cumulative Returns Chart
    fig.add_trace(
        go.Scatter(
            x=dates, y=cumulative_strategy,
            mode='lines', name='Model Strategy',
            line=dict(color=COLORS['strategy'], width=2),
            fill='tozeroy', fillcolor='rgba(46, 204, 113, 0.1)',
            hovertemplate='<b>Date:</b> %{x}<br><b>Strategy:</b> %{y:.1f}%<extra></extra>'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=dates, y=cumulative_benchmark,
            mode='lines', name='Buy & Hold',
            line=dict(color=COLORS['benchmark'], width=2, dash='dash'),
            hovertemplate='<b>Date:</b> %{x}<br><b>Benchmark:</b> %{y:.1f}%<extra></extra>'
        ),
        row=1, col=1
    )
    
    # 2. Rolling Sharpe Ratio
    fig.add_trace(
        go.Scatter(
            x=dates, y=rolling_sharpe,
            mode='lines', name='Rolling Sharpe',
            line=dict(color=COLORS['primary'], width=2),
            fill='tozeroy',
            fillcolor='rgba(30, 136, 229, 0.2)',
            hovertemplate='<b>Date:</b> %{x}<br><b>Sharpe Ratio:</b> %{y:.2f}<extra></extra>'
        ),
        row=1, col=2
    )
    
    # Add zero line
    fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5, row=1, col=2)
    
    # 3. Drawdown Analysis
    fig.add_trace(
        go.Scatter(
            x=dates, y=drawdown,
            mode='lines', name='Drawdown',
            line=dict(color=COLORS['drawdown'], width=2),
            fill='tozeroy', fillcolor='rgba(231, 76, 60, 0.3)',
            hovertemplate='<b>Date:</b> %{x}<br><b>Drawdown:</b> %{y:.1f}%<extra></extra>'
        ),
        row=2, col=1
    )
    
    # 4. Return Distribution (Histogram)
    fig.add_trace(
        go.Histogram(
            x=y_pred * 100, nbinsx=50, name='Strategy Returns',
            marker_color=COLORS['strategy'], opacity=0.7,
            histnorm='probability density',
            hovertemplate='Return: %{x:.2f}%<br>Density: %{y:.3f}<extra></extra>'
        ),
        row=2, col=2
    )
    
    # Add normal distribution overlay
    x_norm = np.linspace(y_pred.min() * 100, y_pred.max() * 100, 100)
    y_norm = stats.norm.pdf(x_norm / 100, y_pred.mean(), y_pred.std()) / 100 * len(y_pred)
    fig.add_trace(
        go.Scatter(
            x=x_norm, y=y_norm,
            mode='lines', name='Normal Fit',
            line=dict(color=COLORS['neutral'], width=2, dash='dash'),
            hovertemplate='Return: %{x:.2f}%<br>Density: %{y:.3f}<extra></extra>'
        ),
        row=2, col=2
    )
    
    # 5. Prediction vs Actual Scatter
    fig.add_trace(
        go.Scatter(
            x=y_true * 100, y=y_pred * 100,
            mode='markers', name='Predictions',
            marker=dict(size=6, color=COLORS['primary'], opacity=0.6,
                       line=dict(width=0.5, color='white')),
            hovertemplate='<b>Actual:</b> %{x:.2f}%<br><b>Predicted:</b> %{y:.2f}%<extra></extra>'
        ),
        row=3, col=1
    )
    
    # Perfect prediction line
    min_val = min(y_true.min(), y_pred.min()) * 100
    max_val = max(y_true.max(), y_pred.max()) * 100
    fig.add_trace(
        go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val],
            mode='lines', name='Perfect Fit',
            line=dict(color=COLORS['drawdown'], width=2, dash='dash'),
            hovertemplate='Perfect Prediction<extra></extra>'
        ),
        row=3, col=1
    )
    
    # 6. Error Distribution
    residuals = (y_true - y_pred) * 100
    fig.add_trace(
        go.Histogram(
            x=residuals, nbinsx=50, name='Residuals',
            marker_color=COLORS['secondary'], opacity=0.7,
            histnorm='probability density',
            hovertemplate='Residual: %{x:.2f}%<br>Density: %{y:.3f}<extra></extra>'
        ),
        row=3, col=2
    )
    
    # Add zero line
    fig.add_vline(x=0, line_dash="dash", line_color="red", opacity=0.7, row=3, col=2)
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f'<b>{title}</b><br><sub>Interactive performance dashboard with real-time metrics</sub>',
            x=0.5,
            font=dict(size=18, family='Arial Black')
        ),
        height=1000,
        width=1300,
        showlegend=True,
        hovermode='closest',
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    
    # Update axes labels
    fig.update_xaxes(title_text='Date', row=1, col=1)
    fig.update_yaxes(title_text='Cumulative Return (%)', row=1, col=1, tickformat='.0%')
    fig.update_xaxes(title_text='Date', row=1, col=2)
    fig.update_yaxes(title_text='Sharpe Ratio', row=1, col=2)
    fig.update_xaxes(title_text='Date', row=2, col=1)
    fig.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
    fig.update_xaxes(title_text='Return (%)', row=2, col=2)
    fig.update_yaxes(title_text='Density', row=2, col=2)
    fig.update_xaxes(title_text='Actual Return (%)', row=3, col=1)
    fig.update_yaxes(title_text='Predicted Return (%)', row=3, col=1)
    fig.update_xaxes(title_text='Residual (%)', row=3, col=2)
    fig.update_yaxes(title_text='Density', row=3, col=2)
    
    # Add range slider to equity curve
    fig.update_xaxes(rangeslider_visible=True, row=1, col=1)
    
    # Add performance summary as annotations
    fig.add_annotation(
        x=1.02, y=0.98, xref='paper', yref='paper',
        text=f'<b>Performance Summary</b><br>' +
             f'Total Return (Strategy): <b>{total_return_strategy:.1f}%</b><br>' +
             f'Total Return (Benchmark): <b>{total_return_benchmark:.1f}%</b><br>' +
             f'Sharpe Ratio: <b>{sharpe_ratio:.2f}</b><br>' +
             f'Max Drawdown: <b>{max_drawdown:.1f}%</b><br>' +
             f'Win Rate: <b>{win_rate:.1f}%</b>',
        showarrow=False,
        font=dict(size=11),
        align='left',
        bordercolor=COLORS['primary'],
        borderwidth=1,
        borderpad=10,
        bgcolor='rgba(255,255,255,0.9)'
    )
    
    # Save files
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved equity curve to {save_path}")
    
    fig.write_html(HTML_PATH / 'equity_curve_dashboard.html')
    
    print(f"✅ Interactive dashboard saved to {HTML_PATH / 'equity_curve_dashboard.html'}")
    
    return fig


def generate_animated_prediction_vs_actual(y_true, y_pred, save_path=None):
    """
    Generate interactive prediction vs actual scatter plot with residuals
    
    Parameters:
    - y_true: actual returns array
    - y_pred: predicted returns array
    - save_path: path to save the plot
    """
    print("\n📊 Generating animated prediction vs actual dashboard...")
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    residuals = y_true - y_pred
    
    # Calculate metrics
    r2 = 1 - np.sum(residuals**2) / np.sum((y_true - np.mean(y_true))**2)
    mae = np.mean(np.abs(residuals))
    rmse = np.sqrt(np.mean(residuals**2))
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Prediction vs Actual Scatter', 'Residuals Distribution',
                       'Residuals vs Fitted', 'Q-Q Plot'),
        specs=[[{'type': 'scatter'}, {'type': 'histogram'}],
               [{'type': 'scatter'}, {'type': 'scatter'}]]
    )
    
    # 1. Scatter plot with color gradient
    fig.add_trace(
        go.Scatter(
            x=y_true * 100, y=y_pred * 100,
            mode='markers',
            marker=dict(
                size=8,
                color=np.abs(residuals),
                colorscale='RdYlBu_r',
                showscale=True,
                colorbar=dict(title='Error', x=1.02),
                opacity=0.6,
                line=dict(width=0.5, color='white')
            ),
            text=[f'Actual: {a:.2f}%<br>Predicted: {p:.2f}%<br>Error: {e:.2f}%' 
                  for a, p, e in zip(y_true * 100, y_pred * 100, residuals * 100)],
            hovertemplate='%{text}<extra></extra>',
            name='Predictions'
        ),
        row=1, col=1
    )
    
    # Perfect prediction line
    min_val = min(y_true.min(), y_pred.min()) * 100
    max_val = max(y_true.max(), y_pred.max()) * 100
    fig.add_trace(
        go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val],
            mode='lines', name='Perfect Prediction',
            line=dict(color='red', width=2, dash='dash'),
            hovertemplate='Perfect Fit<extra></extra>'
        ),
        row=1, col=1
    )
    
    # 2. Residuals histogram with KDE
    fig.add_trace(
        go.Histogram(
            x=residuals * 100, nbinsx=50, name='Residuals',
            marker_color=COLORS['secondary'], opacity=0.7,
            histnorm='probability density',
            hovertemplate='Residual: %{x:.2f}%<br>Density: %{y:.3f}<extra></extra>'
        ),
        row=1, col=2
    )
    
    # Add KDE line
    kde = stats.gaussian_kde(residuals)
    x_range = np.linspace(residuals.min(), residuals.max(), 100) * 100
    fig.add_trace(
        go.Scatter(
            x=x_range, y=kde(residuals) * len(residuals),
            mode='lines', name='KDE',
            line=dict(color=COLORS['primary'], width=2),
            hovertemplate='Residual: %{x:.2f}%<br>Density: %{y:.3f}<extra></extra>'
        ),
        row=1, col=2
    )
    
    # Add zero line
    fig.add_vline(x=0, line_dash="dash", line_color="red", opacity=0.7, row=1, col=2)
    
    # 3. Residuals vs Fitted
    fig.add_trace(
        go.Scatter(
            x=y_pred * 100, y=residuals * 100,
            mode='markers',
            marker=dict(size=6, color=COLORS['primary'], opacity=0.6),
            hovertemplate='<b>Predicted:</b> %{x:.2f}%<br><b>Residual:</b> %{y:.2f}%<extra></extra>',
            name='Residuals'
        ),
        row=2, col=1
    )
    
    # Add zero line
    fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.7, row=2, col=1)
    
    # 4. Q-Q Plot
    sorted_residuals = np.sort(residuals)
    theoretical_quantiles = stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_residuals)))
    
    fig.add_trace(
        go.Scatter(
            x=theoretical_quantiles * 100, y=sorted_residuals * 100,
            mode='markers',
            marker=dict(size=6, color=COLORS['primary'], opacity=0.6),
            name='Q-Q Points',
            hovertemplate='<b>Theoretical:</b> %{x:.2f}%<br><b>Sample:</b> %{y:.2f}%<extra></extra>'
        ),
        row=2, col=2
    )
    
    # Reference line
    qq_slope, qq_intercept = np.polyfit(theoretical_quantiles, sorted_residuals, 1)
    qq_line = qq_slope * theoretical_quantiles + qq_intercept
    fig.add_trace(
        go.Scatter(
            x=theoretical_quantiles * 100, y=qq_line * 100,
            mode='lines', name='Reference',
            line=dict(color='red', width=2, dash='dash'),
            hovertemplate='Reference Line<extra></extra>'
        ),
        row=2, col=2
    )
    
    fig.update_layout(
        title=dict(
            text=f'<b>Prediction vs Actual Analysis</b><br><sub>R²: {r2:.4f} | MAE: {mae*100:.2f}% | RMSE: {rmse*100:.2f}%</sub>',
            x=0.5,
            font=dict(size=16)
        ),
        height=800,
        width=1100,
        showlegend=True,
        template='plotly_white'
    )
    
    fig.update_xaxes(title_text='Actual Return (%)', row=1, col=1)
    fig.update_yaxes(title_text='Predicted Return (%)', row=1, col=1)
    fig.update_xaxes(title_text='Residual (%)', row=1, col=2)
    fig.update_yaxes(title_text='Density', row=1, col=2)
    fig.update_xaxes(title_text='Predicted Return (%)', row=2, col=1)
    fig.update_yaxes(title_text='Residual (%)', row=2, col=1)
    fig.update_xaxes(title_text='Theoretical Quantiles (%)', row=2, col=2)
    fig.update_yaxes(title_text='Sample Quantiles (%)', row=2, col=2)
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved prediction vs actual to {save_path}")
    
    fig.write_html(HTML_PATH / 'prediction_vs_actual.html')
    
    print(f"✅ Interactive dashboard saved to {HTML_PATH / 'prediction_vs_actual.html'}")
    
    return fig


def generate_all_visualizations(y_true, y_pred, output_dir=None):
    """
    Generate all equity curve visualizations
    
    Parameters:
    - y_true: actual returns
    - y_pred: predicted returns
    - output_dir: output directory (optional)
    """
    print("\n" + "="*60)
    print("🎨 Generating Complete Equity Curve Visualizations")
    print("="*60)
    
    # Generate main equity curve dashboard
    generate_animated_equity_curve(
        y_true, y_pred,
        title="S&P 500 Model Performance Dashboard",
        save_path=HTML_PATH / 'equity_curve_complete.html'
    )
    
    # Generate prediction vs actual dashboard
    generate_animated_prediction_vs_actual(
        y_true, y_pred,
        save_path=HTML_PATH / 'prediction_analysis.html'
    )
    
    print("\n" + "="*60)
    print("✅ All visualizations generated successfully!")
    print(f"📁 Output directory: {HTML_PATH}")
    print("="*60)


if __name__ == "__main__":
    # Demo with sample data
    np.random.seed(42)
    n_samples = 500
    
    # Generate realistic returns
    y_true = np.random.normal(0.001, 0.02, n_samples)
    y_pred = y_true + np.random.normal(0, 0.008, n_samples)
    
    # Generate all visualizations
    generate_all_visualizations(y_true, y_pred)


🎨 Generating Complete Equity Curve Visualizations

📈 Generating animated equity curve dashboard...
✅ Saved equity curve to visualizations\equity\html\equity_curve_complete.html
✅ Interactive dashboard saved to visualizations\equity\html\equity_curve_dashboard.html

📊 Generating animated prediction vs actual dashboard...
✅ Saved prediction vs actual to visualizations\equity\html\prediction_analysis.html
✅ Interactive dashboard saved to visualizations\equity\html\prediction_vs_actual.html

✅ All visualizations generated successfully!
📁 Output directory: visualizations\equity\html
